# Chapter 5 — Bit Manipulation

**Python caveat:** Python ints are arbitrary precision, so there is no fixed 32-bit width and no overflow. Negative numbers behave as if they had infinitely many leading 1s (two's complement). When a problem assumes 32-bit ints, mask with `& 0xFFFFFFFF` and convert back with a helper.

Common operations: get/set/clear/update bit, `x & (x - 1)` clears the lowest set bit, `x & -x` isolates it.

In [1]:
MASK32 = 0xFFFFFFFF


def to_signed32(x: int) -> int:
    x &= MASK32
    return x - (1 << 32) if x & (1 << 31) else x


def get_bit(num: int, i: int) -> bool:
    return (num >> i) & 1 == 1


def set_bit(num: int, i: int) -> int:
    return num | (1 << i)


def clear_bit(num: int, i: int) -> int:
    return num & ~(1 << i)


def clear_bits_msb_through_i(num: int, i: int) -> int:
    return num & ((1 << i) - 1)


def clear_bits_i_through_0(num: int, i: int) -> int:
    return num & (-1 << (i + 1))


def update_bit(num: int, i: int, bit_is_1: bool) -> int:
    return (num & ~(1 << i)) | (int(bit_is_1) << i)


n = 0b10110
assert get_bit(n, 1) and not get_bit(n, 0)
assert set_bit(n, 0) == 0b10111 and clear_bit(n, 1) == 0b10100
assert clear_bits_msb_through_i(n, 3) == 0b110
assert clear_bits_i_through_0(n, 2) == 0b10000
assert update_bit(n, 0, True) == 0b10111
assert to_signed32(0xFFFFFFFF) == -1
print(f"{-8 & MASK32:032b}  <- -8 as 32-bit two's complement")
print("bit helpers ok")

11111111111111111111111111111000  <- -8 as 32-bit two's complement
bit helpers ok


## 5.1 Insertion
Given 32-bit N and M, and positions i < j, insert M into N so M occupies bits j..i. (Assume the bits fit.)

In [2]:
def insert_bits(n: int, m: int, i: int, j: int) -> int:
    mask = ~(((1 << (j - i + 1)) - 1) << i)  # zeros at j..i, ones elsewhere
    return (n & mask) | (m << i)


assert insert_bits(0b10000000000, 0b10011, 2, 6) == 0b10001001100
print("5.1 ok")

5.1 ok


## 5.2 Binary to String
Given a real number in (0, 1) as a double, print its binary representation, or `ERROR` if it can't be represented within 32 characters. Multiply by 2 repeatedly; the integer part is the next bit.

In [3]:
def binary_to_string(num: float) -> str:
    if not 0 < num < 1:
        return "ERROR"
    parts = ["0."]
    while num > 0:
        if len("".join(parts)) >= 32:
            return "ERROR"
        num *= 2
        if num >= 1:
            parts.append("1")
            num -= 1
        else:
            parts.append("0")
    return "".join(parts)


assert binary_to_string(0.625) == "0.101"
assert binary_to_string(0.72) == "ERROR"
print("5.2 ok")

5.2 ok


## 5.3 Flip Bit to Win
You may flip exactly one 0 to 1. Return the longest run of 1s you can create. Track the current run and the previous run (reset previous if we see two 0s in a row). `O(b)` time, `O(1)` space.

In [4]:
def flip_bit_to_win(n: int, bits: int = 32) -> int:
    n &= (1 << bits) - 1
    if n == (1 << bits) - 1:
        return bits
    current = previous = 0
    best = 1
    for _ in range(bits):
        if n & 1:
            current += 1
        else:
            previous = current if n & 2 else 0
            current = 0
        best = max(best, previous + current + 1)
        n >>= 1
    return best


assert flip_bit_to_win(1775) == 8  # 11011101111
assert flip_bit_to_win(0) == 1
assert flip_bit_to_win(-1) == 32
print("5.3 ok")

5.3 ok


## 5.4 Next Number
Given a positive int, return the next largest and next smallest numbers with the same count of 1 bits.

- **Next larger**: find the rightmost non-trailing zero `p` (a 0 with 1s to its right), flip it to 1, clear bits below `p`, then put `c1 - 1` ones at the bottom.
- **Next smaller**: find the rightmost non-trailing one `p` (a 1 with 0s to its right), flip it to 0, and pack `c1 + 1` ones immediately below it.

In [5]:
def next_larger(n: int) -> int:
    c, c0, c1 = n, 0, 0
    while c & 1 == 0 and c != 0:
        c0 += 1; c >>= 1
    while c & 1 == 1:
        c1 += 1; c >>= 1
    if c0 + c1 == 31 or c0 + c1 == 0:
        return -1
    p = c0 + c1
    n |= 1 << p
    n &= ~((1 << p) - 1)
    n |= (1 << (c1 - 1)) - 1
    return n


def next_smaller(n: int) -> int:
    c, c0, c1 = n, 0, 0
    while c & 1 == 1:
        c1 += 1; c >>= 1
    if c == 0:
        return -1
    while c & 1 == 0 and c != 0:
        c0 += 1; c >>= 1
    p = c0 + c1
    n &= -1 << (p + 1)  # clear bits p..0
    mask = (1 << (c1 + 1)) - 1
    n |= mask << (c0 - 1)
    return n


def brute_next(n: int, step: int) -> int:
    target = n.bit_count()
    m = n + step
    while m > 0:
        if m.bit_count() == target:
            return m
        m += step
    return -1


for x in range(1, 2000):
    assert next_larger(x) == brute_next(x, 1), x
    assert next_smaller(x) == brute_next(x, -1), x
print(f"13948 -> larger {next_larger(13948)}, smaller {next_smaller(13948)}")
print("5.4 ok")

13948 -> larger 13967, smaller 13946
5.4 ok


## 5.5 Debugger
What does `(n & (n - 1)) == 0` check? `n - 1` flips the lowest 1 and everything after it; ANDing clears the lowest 1. The result is 0 only if `n` had **at most one** 1 bit → `n` is a power of two (or zero).

In [6]:
def is_power_of_two_or_zero(n: int) -> bool:
    return n & (n - 1) == 0


assert [x for x in range(40) if is_power_of_two_or_zero(x)] == [0, 1, 2, 4, 8, 16, 32]
print("5.5 ok")

5.5 ok


## 5.6 Conversion
How many bits must change to turn integer A into B? Count the 1s in `A ^ B` (using `c & (c - 1)` to drop one 1 per step, or `int.bit_count()` in Python 3.10+).

In [7]:
def bits_to_convert(a: int, b: int) -> int:
    c = (a ^ b) & MASK32
    count = 0
    while c:
        c &= c - 1
        count += 1
    return count


assert bits_to_convert(29, 15) == 2
assert bits_to_convert(-1, 0) == 32
assert ((29 ^ 15).bit_count()) == 2
print("5.6 ok")

5.6 ok


## 5.7 Pairwise Swap
Swap odd and even bits using as few instructions as possible. Mask with `0xAAAAAAAA` (odd positions) / `0x55555555` (even), shift, OR.

In [8]:
def pairwise_swap(x: int) -> int:
    return ((x & 0xAAAAAAAA) >> 1) | ((x & 0x55555555) << 1)


assert pairwise_swap(0b10) == 0b01
assert pairwise_swap(0b100110) == 0b011001
print("5.7 ok")

5.7 ok


## 5.8 Draw Line
A monochrome screen is stored as a flat byte array, 8 pixels per byte, `width` pixels per row (divisible by 8). Draw a horizontal line from `(x1, y)` to `(x2, y)`. Set full bytes directly, then mask the partial bytes at each end.

In [9]:
def draw_line(screen: bytearray, width: int, x1: int, x2: int, y: int) -> None:
    start_offset, first_full = x1 % 8, x1 // 8
    if start_offset:
        first_full += 1
    end_offset, last_full = x2 % 8, x2 // 8
    if end_offset != 7:
        last_full -= 1
    row = (width // 8) * y
    for b in range(first_full, last_full + 1):
        screen[row + b] = 0xFF
    start_mask = 0xFF >> start_offset
    end_mask = ~(0xFF >> (end_offset + 1)) & 0xFF
    if x1 // 8 == x2 // 8:  # same byte
        screen[row + x1 // 8] |= start_mask & end_mask
    else:
        if start_offset:
            screen[row + x1 // 8] |= start_mask
        if end_offset != 7:
            screen[row + x2 // 8] |= end_mask


def render(screen: bytearray, width: int) -> str:
    bytes_per_row = width // 8
    return "\n".join(
        "".join(f"{screen[r * bytes_per_row + b]:08b}" for b in range(bytes_per_row))
        for r in range(len(screen) // bytes_per_row)
    )


W = 32
for x1, x2 in [(3, 28), (2, 5), (0, 31), (8, 15), (7, 8)]:
    scr = bytearray(W // 8 * 2)
    draw_line(scr, W, x1, x2, 1)
    row = render(scr, W).splitlines()[1]
    assert row == "0" * x1 + "1" * (x2 - x1 + 1) + "0" * (W - 1 - x2), (x1, x2, row)
scr = bytearray(W // 8 * 3)
draw_line(scr, W, 3, 28, 1)
print(render(scr, W))
print("5.8 ok")

00000000000000000000000000000000
00011111111111111111111111111000
00000000000000000000000000000000
5.8 ok
